# Incompressible OpenFOAM Baseline: Airfoil

## Preamble

In [1]:
# Standard Library
import os
from pathlib import Path
import shutil
import subprocess
from tempfile import mkdtemp

# FoamNordic
import foamnordic as fno

### Directories

In [2]:
# Set the case and output directories
tutorials_dir = os.environ.get("FOAM_TUTORIALS")
if os.environ.get("WM_PROJECT_VERSION") != "14" or not tutorials_dir:
    raise RuntimeError("Load openfoam-org/14 before starting the notebook kernel")

SOURCE_CASE = Path(tutorials_dir) / "incompressibleFluid/airFoil2D"
if not (SOURCE_CASE / "constant/polyMesh").is_dir():
    raise FileNotFoundError(f"The official airFoil2D mesh is missing from {SOURCE_CASE}")

OUTPUT_DIR = Path("output/airfoil2d-of14").resolve()

### Configuration

In [3]:
# HPC configuration
ACCOUNT = "<allocation-account>"  # e.g. project_xxxxxxx
PARTITION = "test"
TIME = "00:05:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "4G"

if ACCOUNT == "<allocation-account>":
    raise ValueError("Set ACCOUNT to your Slurm allocation account")

# OpenFOAM configuration
OF_CMD = "openfoam-org/14"
N_ITERATIONS = 5

In [4]:
# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=None,
)

## Example - OpenFOAM on FoamNordic (Baseline)

### Case Definition

In [5]:
# Shorten the copied case and write the final iteration
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
case_dir = Path(mkdtemp(prefix="airFoil2D-", dir=OUTPUT_DIR)) / "airFoil2D"
shutil.copytree(SOURCE_CASE, case_dir)
control_dict = case_dir / "system/controlDict"

for entry in ("endTime", "writeInterval"):
    subprocess.run(
        ["foamDictionary", str(control_dict), "-entry", entry, "-set", str(N_ITERATIONS)],
        check=True,
        capture_output=True,
        text=True,
    )

In [6]:
# Initialize the OpenFOAM case with the supplied mesh
case = fno.OpenFOAM.Case(
    case_dir=case_dir,
    run_dir=OUTPUT_DIR,
    of_cmd=OF_CMD,
    shell="bash",
    application="foamRun",
    ranks=N_TASKS,
)
case.initialize(validate_mesh=True)

### Submit Job

In [7]:
# Connect the OpenFOAM case and execution resources
longship = fno.Longship(case=case, scheduler=scheduler, verbose=True)

Resource,OpenFOAM,Model,Allocation
Processes,1 tasks,0 host(s),1 node(s)
CPU,1 cores,0 cores,1 cores
Memory,4 GiB,0 KiB,4 GiB
Field programs,-,0,0
SHM,-,none,none


In [8]:
# Launch the workload
run = longship.launch()

[FoamNordic] Validating existing mesh with checkMesh: airFoil2D


[FoamNordic] Mesh is ready: airFoil2D


[FoamNordic] Sailing submitted with Job ID: 3416545


[FoamNordic] Sailing has launched with Job ID: 3416545
[FoamNordic] Sailing started at: 2026-11-25T15:26:41
[FoamNordic] Sailing in background: airFoil2D


In [9]:
# Wait for the job to complete
result = run.stop(timeout=600, progress=True)
if not result.success:
    raise RuntimeError(f"OpenFOAM failed; inspect {result.solver_log}")

In [10]:
# Summary of the job result
result.summary(style="compact", display=True)

Job ID,Name,Status,Partition,Node,Elapsed
3416545,airFoil2D,succeeded,interactive,rc4183,00:00:03


### Post-processing

In [11]:
# Postprocessing
post = result.postprocess
statistics = post.statistics(fields=["U", "p"], time_idx=-1, verbose=True)

Field,Min,Max,Mean,Std,RMS
U,7.671460e-01,3.664675e+01,2.593697e+01,2.444439e+00,2.605191e+01
p,-1.249520e+03,4.175170e+02,-4.592627e+01,1.616952e+02,1.680909e+02
